In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1: Extreme ESI 1 Binary Feature Engineered Logistic Regressor (`models/lr_feng_esi1_extreme.ipynb`)

This notebook trains a **Binary Logistic Regressor** to classify **ESI 1 (Highest Acuity) vs. Not ESI 1** and includes **Precision, Recall & PR-AUC Diagnostics**:
- **Binary Target Output**: Predicts whether a patient is **ESI 1** (`"1"`) or **Not ESI 1** (`"not_1"`).
- **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and the **10 Clinical Feature Engineering flags** defined in `TODO.md` (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
- **Individual Metrics**: Reports **Accuracy**, **Precision**, **Recall (Sensitivity)**, **PR-AUC**, and **Log Loss** across Train, Validation, and Test sets.
- **Configurable Target Class Subsampling**:
  - `keep_ratio_1 = 1.00` (Keeps 100% of ESI 1 rows).
  - `keep_ratio_not_1 = 0.005` (Keeps 0.5% of 'not_1' rows, configurable).
- **Overfit / Underfit Diagnostics (Individual Cell Plots + PNG Artifacts)**:
  - **Plot 1: Metrics Comparison Bar Chart** (Accuracy, Precision, Recall, PR-AUC).
  - **Plot 2: Overlaid Precision-Recall (PR) Curves**.
  - **Plot 3: Learning Curve Plot** (PR-AUC vs. Training Sample Size).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute 10 FE Flags + Demographics & Apply Binary Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender + cc_breathingdifficulty
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

df_feng[[target_col]] <- raw_df[[target_col]]

# Create Binary ESI 1 Target: '1' vs 'not_1'
raw_esi <- as.character(df_feng[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"),
                                levels = c("1", "not_1"))

# ---------------------------------------------------------
# Subsample Binary Target Classes (Configurable keep ratios per class)
# ---------------------------------------------------------
keep_ratio_1     <- 1.00   # Keep 100% of ESI 1 rows
keep_ratio_not_1 <- 0.5  # Keep 0.5% of 'not_1' rows (configurable)

idx_1     <- which(df_feng$target_layer1 == "1")
idx_not_1 <- which(df_feng$target_layer1 == "not_1")

kept_1     <- sample(idx_1,     size = round(length(idx_1)     * keep_ratio_1))
kept_not_1 <- sample(idx_not_1, size = round(length(idx_not_1) * keep_ratio_not_1))

df_feng <- df_feng[sort(c(kept_1, kept_not_1)), ]

cat(sprintf("Binary ESI 1 FE Dataset Ready (Class Ratios: ESI 1=%.0f%%, Not ESI 1=%.2f%%): %d rows x %d cols\n",
            keep_ratio_1 * 100, keep_ratio_not_1 * 100, nrow(df_feng), ncol(df_feng)))
cat("Feature Engineered Input Columns (13):\n", paste(setdiff(names(df_feng), c(target_col, "target_layer1")), collapse = ", "), "\n")
cat("Binary Target Distribution:\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age strictly)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary ESI 1 Feature Engineered Logistic Regressor
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names <- setdiff(names(train_df), c(target_col, "target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))

cat("Training Binary ESI 1 Feature Engineered Logistic Regressor...\n")
lr_esi1_model <- multinom(formula_lr, data = train_df, trace = FALSE, MaxNWts = 5000)

cat("Binary ESI 1 Logistic Regression training complete!\n")
print(summary(lr_esi1_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Evaluation Across Splits (Accuracy, Precision, Recall, PR-AUC, Log Loss)
# ---------------------------------------------------------
# Function to compute PR-AUC (Precision-Recall Area Under Curve)
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  if (!is.matrix(prob_matrix)) {
    prob_matrix <- cbind(1 - prob_matrix, prob_matrix)
    colnames(prob_matrix) <- levels(actual_factor)
  }
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}
evaluate_esi1_lr <- function(model, data, set_name) {
  prob_res <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data$target_layer1)
  
  if (is.matrix(prob_res)) {
    prob_matrix <- prob_res
  } else {
    prob_matrix <- cbind(1 - prob_res, prob_res)
    colnames(prob_matrix) <- c("1", "not_1")
  }
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor, positive = "1")
  acc  <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  
  prob_1 <- prob_matrix[, "1"]
  act_binary_1 <- ifelse(actual_factor == "1", 1, 0)
  pr_auc <- calc_pr_auc(act_binary_1, prob_1)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 1 FEATURE ENGINEERED LR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Precision            : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  Recall (Sensitivity) : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  PR-AUC               : %.4f\n", pr_auc))
  cat(sprintf("  Log Loss             : %.4f\n", log_loss))
  cat("\nTarget Class Counts (Actual vs Predicted Comparison):\n")
  class_counts_df <- data.frame(
    Class = target_classes,
    Actual_Count = as.numeric(table(actual_factor)[target_classes]),
    Predicted_Count = as.numeric(table(pred_factor)[target_classes]),
    Diff = as.numeric(table(pred_factor)[target_classes]) - as.numeric(table(actual_factor)[target_classes])
  )
  print(class_counts_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, pr_auc = pr_auc, log_loss = log_loss, prob_matrix = prob_matrix, actual_factor = actual_factor))
}

# Evaluate on Train, Validation, and Test Sets
res_train <- evaluate_esi1_lr(lr_esi1_model, train_df, "Train")
res_val   <- evaluate_esi1_lr(lr_esi1_model, val_df,   "Validation")
res_test  <- evaluate_esi1_lr(lr_esi1_model, test_df,  "Test")

# Compare Metrics Summary Across Splits
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,  res_val$acc,  res_test$acc),
  Precision = c(res_train$prec, res_val$prec, res_test$prec),
  Recall    = c(res_train$rec,  res_val$rec,  res_test$rec),
  PR_AUC    = c(res_train$pr_auc, res_val$pr_auc, res_test$pr_auc),
  Log_Loss  = c(res_train$log_loss, res_val$log_loss, res_test$log_loss)
)

cat("=== OVERALL METRICS COMPARISON (TRAIN vs VALIDATION vs TEST) ===\n")
print(metrics_summary)

# Diagnose Overfitting / Underfitting
delta_acc <- res_train$acc - res_val$acc
delta_pr_auc <- res_train$pr_auc - res_val$pr_auc

cat("\n=== DIAGNOSTIC EVALUATION SUMMARY ===\n")
cat(sprintf("  - Accuracy Delta (Train - Val):  %+.4f\n", delta_acc))
cat(sprintf("  - PR-AUC Delta (Train - Val):    %+.4f\n", delta_pr_auc))
if (!is.na(delta_pr_auc) && delta_pr_auc > 0.05) {
  cat("  -> DIAGNOSIS: Potential OVERFITTING detected (Train PR-AUC is significantly higher than Validation).\n")
} else if (res_val$acc < 0.60 && res_train$acc < 0.60) {
  cat("  -> DIAGNOSIS: Potential UNDERFITTING detected (Low accuracy on both Train & Validation).\n")
} else {
  cat("  -> DIAGNOSIS: WELL-GENERALIZED MODEL (Train, Validation, and Test metrics are closely aligned).\n")
}

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6A: Diagnostic Plot 1 - Metrics Comparison Bar Chart (Accuracy, Precision, Recall, PR-AUC)
# ---------------------------------------------------------
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")

p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison (Binary ESI 1)",
       subtitle = "Comparing Accuracy, Precision, Recall, and PR-AUC across splits",
       y = "Metric Value", x = "") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "top")

if (!dir.exists("../plots")) dir.create("../plots", recursive = TRUE)
ggsave("../plots/esi1_metrics_comparison_barchart.png", plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Bar Chart saved to: plots/esi1_metrics_comparison_barchart.png\n")

p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6B: Diagnostic Plot 2 - Overlaid Precision-Recall (PR) Curves
# ---------------------------------------------------------
get_pr_df <- function(actual_factor, prob_positive, split_label) {
  ord <- order(prob_positive, decreasing = TRUE)
  act_sorted <- (actual_factor[ord] == "1")
  tp <- cumsum(act_sorted)
  fp <- cumsum(!act_sorted)
  n_pos <- sum(act_sorted)
  rec  <- c(0, tp / n_pos)
  prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
  pr_auc_val <- calc_pr_auc(ifelse(actual_factor == "1", 1, 0), prob_positive)
  data.frame(
    Recall    = rec,
    Precision = prec,
    Split     = sprintf("%s (PR-AUC = %.3f)", split_label, pr_auc_val)
  )
}

df_pr_tr <- get_pr_df(res_train$actual_factor, res_train$prob_matrix[, "1"], "Train")
df_pr_va <- get_pr_df(res_val$actual_factor,   res_val$prob_matrix[, "1"],   "Validation")
df_pr_te <- get_pr_df(res_test$actual_factor,  res_test$prob_matrix[, "1"],  "Test")

df_pr_all <- rbind(df_pr_tr, df_pr_va, df_pr_te)

p_pr <- ggplot(df_pr_all, aes(x = Recall, y = Precision, color = Split)) +
  geom_line(size = 1.2) +
  theme_minimal() +
  scale_color_manual(values = c("#2b5c8f", "#e07a5f", "#81b29a")) +
  labs(title = "Precision-Recall (PR) Curves Comparison Across Splits",
       subtitle = "Overlaid PR curves to evaluate precision & recall trade-off for ESI 1",
       x = "Recall (Sensitivity)", y = "Precision (Positive Predictive Value)") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "bottom")

ggsave("../plots/esi1_pr_curves_overlaid.png", plot = p_pr, width = 8, height = 5, dpi = 300)
cat("PR Curves plot saved to: plots/esi1_pr_curves_overlaid.png\n")

p_pr

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6C: Diagnostic Plot 3 - Learning Curve (Sample Size vs PR-AUC)
# ---------------------------------------------------------
sample_fractions <- c(0.2, 0.4, 0.6, 0.8, 1.0)
lc_results <- data.frame()

set.seed(config$training$random_state)
for (frac in sample_fractions) {
  n_sub <- round(nrow(train_df) * frac)
  sub_idx <- sample(1:nrow(train_df), size = n_sub)
  sub_train <- train_df[sub_idx, ]
  
  fit_sub <- multinom(formula_lr, data = sub_train, trace = FALSE, MaxNWts = 5000)
  
  tr_prob <- predict(fit_sub, newdata = sub_train, type = "probs")
  va_prob <- predict(fit_sub, newdata = val_df,   type = "probs")
  
  tr_prob_1 <- if (is.matrix(tr_prob)) tr_prob[, "1"] else tr_prob
  va_prob_1 <- if (is.matrix(va_prob)) va_prob[, "1"] else va_prob
  
  tr_prauc <- calc_pr_auc(ifelse(sub_train$target_layer1 == "1", 1, 0), tr_prob_1)
  va_prauc <- calc_pr_auc(ifelse(val_df$target_layer1 == "1", 1, 0), va_prob_1)
  
  lc_results <- rbind(lc_results, data.frame(Fraction = frac * 100, Train_PR_AUC = tr_prauc, Val_PR_AUC = va_prauc))
}

lc_long <- lc_results %>%
  pivot_longer(cols = c("Train_PR_AUC", "Val_PR_AUC"), names_to = "Split", values_to = "PR_AUC") %>%
  mutate(Split = ifelse(Split == "Train_PR_AUC", "Train", "Validation"))

p_lc <- ggplot(lc_long, aes(x = Fraction, y = PR_AUC, color = Split, group = Split)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  theme_minimal() +
  scale_color_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f")) +
  labs(title = "Learning Curve: Sample Size vs PR-AUC (Binary ESI 1)",
       subtitle = "Convergence of Train and Validation curves shows sample size adequacy",
       x = "Training Data Percentage (%)", y = "PR-AUC Score") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "top")

ggsave("../plots/esi1_learning_curve.png", plot = p_lc, width = 8, height = 5, dpi = 300)
cat("Learning Curve saved to: plots/esi1_learning_curve.png\n")

p_lc

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Binary ESI 1 Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
saveRDS(list(model = lr_esi1_model, preproc = preproc), file = model_path)
cat("Binary ESI 1 Feature Engineered Logistic Regressor model saved to:", model_path, "\n")